# 2.3 — Gradient Descent

Gradient descent turns local slope information into repeated improvement: compute the gradient, step in the opposite direction, and let many small moves replace an impossible exact solve of $\nabla f(x)=0$. This lesson builds the update from scratch, shows why the learning rate matters, and connects the same idea to regression, feature scaling, and convergence diagnostics.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build gradient descent one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the update is never a black box. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, vector arithmetic, and small numerical checks.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for random starting points.

### 1. The gradient is the local uphill direction

Gradient descent starts with one local fact: the derivative tells us which way the function increases fastest. For a one-dimensional function $f(x)=(x-3)^2$, the derivative is $f'(x)=2(x-3)$. If $x=0$, the derivative is negative, which means increasing $x$ moves downhill even though the derivative itself points uphill toward decreasing $x$.

In [ ]:
x_grid_w = np.linspace(-1, 7, 200)  # points for drawing the quadratic bowl.
f_grid_w = (x_grid_w - 3) ** 2  # objective values on the grid.
x0_w = 0.0  # current location.
g0_w = 2 * (x0_w - 3)  # derivative of (x-3)^2 at x0.
print("x0:", x0_w, "gradient:", g0_w, "negative gradient:", -g0_w)
assert g0_w == -6.0

▶ What you'll see: at `x0=0`, the gradient is `-6`, so the descent direction is `+6`.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(x_grid_w, f_grid_w, color="navy")
plt.scatter([x0_w], [(x0_w - 3) ** 2], color="crimson", zorder=3)
plt.arrow(x0_w, 9.0, -0.7, 0, head_width=0.45, color="crimson", length_includes_head=True, label="gradient")
plt.arrow(x0_w, 7.6, 0.7, 0, head_width=0.45, color="seagreen", length_includes_head=True, label="-gradient")
plt.title("1: gradient points uphill; -gradient descends")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.legend(); plt.show()

▶ What you'll see: the red arrow points left/uphill while the green arrow points right/downhill toward the minimizer.

*Why it's done this way:* the first-order approximation says $f(x+s)\approx f(x)+f'(x)s$. To make that local model smaller, choose $s$ with the opposite sign of $f'(x)$; in many dimensions, the same dot-product logic says $s=-\eta\nabla f(x)$ gives immediate predicted decrease.

### 2. One update: direction and step size are separate

The gradient gives the direction, but the learning rate $\eta$ decides how much we trust that local direction. The update is $x_{t+1}=x_t-\eta\nabla f(x_t)$. On the same quadratic, $x_0=0$ and $\eta=0.1$ produce $x_1=0.6$.

In [ ]:
eta_w = 0.1  # step size: how boldly to trust the local slope.
x1_w = x0_w - eta_w * g0_w  # gradient descent update.
f0_w = (x0_w - 3) ** 2
f1_w = (x1_w - 3) ** 2
print("x1:", x1_w, "f0:", f0_w, "f1:", f1_w)
assert round(x1_w, 3) == 0.6 and round(f1_w, 2) == 5.76

▶ What you'll see: the objective drops from `9.0` to `5.76` after one small step.

In [ ]:
plt.figure(figsize=(4.6, 3))
plt.plot(x_grid_w, f_grid_w, color="navy")
plt.scatter([x0_w, x1_w], [f0_w, f1_w], color=["crimson", "seagreen"], zorder=3)
plt.plot([x0_w, x1_w], [f0_w, f1_w], "--", color="gray")
plt.title("2: one gradient-descent update")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()

▶ What you'll see: the new point moves rightward and downward along the bowl.

*Why it's done this way:* multiplying by $\eta$ separates two decisions that are often confused: the gradient determines the locally best direction, while $\eta$ limits distance because the tangent/linear approximation is only trustworthy nearby.

### 3. Repeating updates accumulates local progress

Gradient descent is useful because one local improvement can be repeated. For $f(x)=(x-3)^2$, the error $e_t=x_t-3$ follows $e_{t+1}=(1-2\eta)e_t$. With $\eta=0.2$, the error multiplies by $0.6$ each step, so the sequence contracts toward the minimizer.

In [ ]:
eta_stable_w = 0.2
xs_w = [0.0]
for t_w in range(20):
    grad_w = 2 * (xs_w[-1] - 3)
    xs_w.append(xs_w[-1] - eta_stable_w * grad_w)
xs_w = np.array(xs_w)
losses_w = (xs_w - 3) ** 2
print("first five x:", np.round(xs_w[:5], 3))
print("final error magnitude:", abs(xs_w[-1] - 3))
assert abs(abs(xs_w[-1] - 3) - 3 * 0.6 ** 20) < 1e-10

▶ What you'll see: `x` moves 0 → 1.2 → 1.92 → 2.352 → ... and the error shrinks geometrically.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(losses_w, marker="o", color="purple")
plt.yscale("log")
plt.title("3: stable steps make loss fall geometrically")
plt.xlabel("iteration"); plt.ylabel("loss, log scale"); plt.show()

▶ What you'll see: the loss curve is nearly a straight descending line on a log scale, the signature of geometric contraction.

*Why it's done this way:* for a quadratic, the update is an exact linear recurrence, so convergence is controlled by $|1-2\eta|<1$. Repetition works because every step keeps shrinking the distance to the optimum instead of merely making one lucky move.

### 4. Step size must respect curvature

The same descent direction can converge or explode depending on $\eta$. For $f(x)=(x-3)^2$, curvature is $L=2$, and constant-step gradient descent is stable for $0<\eta<1$. With $\eta=1.1$, the error multiplier is $-1.2$: the sign flips and the magnitude grows.

In [ ]:
eta_bad_w = 1.1
xs_bad_w = [0.0]
for t_w in range(8):
    grad_bad_w = 2 * (xs_bad_w[-1] - 3)
    xs_bad_w.append(xs_bad_w[-1] - eta_bad_w * grad_bad_w)
xs_bad_w = np.array(xs_bad_w)
print("unstable x path:", np.round(xs_bad_w, 3))
print("final error magnitude:", round(abs(xs_bad_w[-1] - 3), 3))
assert round(abs(xs_bad_w[-1] - 3), 3) == 12.899

▶ What you'll see: the iterates jump across the minimizer with growing distance.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot((xs_w - 3) ** 2, marker="o", label="η=0.2 stable")
plt.plot((xs_bad_w - 3) ** 2, marker="s", label="η=1.1 unstable")
plt.yscale("log")
plt.title("4: learning rate controls stability")
plt.xlabel("iteration"); plt.ylabel("loss, log scale"); plt.legend(); plt.show()

▶ What you'll see: the stable curve falls while the large-step curve rises after oscillating.

*Why it's done this way:* curvature measures how quickly the gradient changes. A large $\eta$ trusts the old gradient too far; once the step overshoots the bowl enough that $|1-\eta L|>1$, local descent turns into global divergence.

### 5. Vector gradients have the same shape as parameters

In more than one dimension, the gradient is a vector of partial derivatives. For $f(w)= (w_0-1)^2+4(w_1+2)^2$, the second coordinate has larger curvature, so the gradient component for $w_1$ can be much larger. The update still subtracts a same-shaped gradient vector.

In [ ]:
w_w = np.array([-2.0, 1.0])
grad_vec_w = np.array([2 * (w_w[0] - 1), 8 * (w_w[1] + 2)])
eta_vec_w = 0.1
w_next_w = w_w - eta_vec_w * grad_vec_w
print("w:", w_w, "gradient:", grad_vec_w, "w_next:", w_next_w)
assert np.allclose(w_next_w, [-1.4, -1.4])

▶ What you'll see: the gradient has two entries, one per parameter, and the steeper coordinate moves more.

In [ ]:
xv_w = np.linspace(-3, 3, 80)
yv_w = np.linspace(-3.5, 2, 80)
Xv_w, Yv_w = np.meshgrid(xv_w, yv_w)
Zv_w = (Xv_w - 1) ** 2 + 4 * (Yv_w + 2) ** 2
plt.figure(figsize=(4.8, 3.6))
plt.contour(Xv_w, Yv_w, Zv_w, levels=18, cmap="viridis")
plt.scatter([w_w[0], w_next_w[0], 1], [w_w[1], w_next_w[1], -2], color=["crimson", "seagreen", "black"])
plt.title("5: vector step on an elliptical bowl")
plt.xlabel("w0"); plt.ylabel("w1"); plt.show()

▶ What you'll see: contours are stretched ellipses, and the update moves sharply in the high-curvature coordinate.

*Why it's done this way:* each partial derivative answers “if I change only this coordinate, how fast does the objective change?” Stacking them gives the direction of steepest increase under Euclidean distance, so subtracting that vector is the most direct local decrease step.

### 6. Regression gradients average evidence from all examples

For linear regression with prediction $\hat y_i=w x_i$ and loss $L(w)=\frac{1}{n}\sum_i (w x_i-y_i)^2$, the derivative is $\frac{1}{n}\sum_i 2x_i(w x_i-y_i)$. At $w=0$ for $x=(1,2,3)$ and $y=2x$, every example says the slope is too small, so the gradient is strongly negative.

In [ ]:
X_reg_w = np.array([1.0, 2.0, 3.0])
y_reg_w = 2 * X_reg_w
w0_reg_w = 0.0
resid_reg_w = w0_reg_w * X_reg_w - y_reg_w
terms_reg_w = 2 * X_reg_w * resid_reg_w
grad_reg_w = terms_reg_w.mean()
print("gradient terms:", terms_reg_w, "mean gradient:", round(grad_reg_w, 3))
assert round(grad_reg_w, 3) == -18.667

▶ What you'll see: terms `[-4, -16, -36]` average to `-18.667`.

In [ ]:
eta_reg_w = 0.05
w1_reg_w = w0_reg_w - eta_reg_w * grad_reg_w
loss0_reg_w = np.mean((w0_reg_w * X_reg_w - y_reg_w) ** 2)
loss1_reg_w = np.mean((w1_reg_w * X_reg_w - y_reg_w) ** 2)
print("w1:", round(w1_reg_w, 3), "loss before/after:", round(loss0_reg_w, 3), round(loss1_reg_w, 3))
assert round(w1_reg_w, 3) == 0.933

▶ What you'll see: the first regression step jumps the slope from `0` to about `0.933` and lowers MSE.

In [ ]:
plt.figure(figsize=(4.8, 3.2))
plt.scatter(X_reg_w, y_reg_w, color="black", label="data")
plt.plot(X_reg_w, w0_reg_w * X_reg_w, "--", label="before")
plt.plot(X_reg_w, w1_reg_w * X_reg_w, label="after one step")
plt.title("6: regression gradient moves the line toward data")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()

▶ What you'll see: the fitted line rotates upward because all residuals agreed the slope was too low.

*Why it's done this way:* the gradient is an average of per-example forces, so examples with larger $x_i$ and larger residuals contribute more. The update is large here because every term points in the same direction: increase $w$.

## 🛠️ Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)

## 🟢 Basics (warm-up)

### Basic 1 — Evaluate a quadratic objective

**Goal.** Compute values of $f(x)=(x-3)^2$, because gradient descent needs an objective whose decrease we can inspect.

In [ ]:
x_b1 = np.array([0.0, 1.0, 3.0, 5.0])
f_b1 = (x_b1 - 3) ** 2
print("x values:", x_b1)
print("f(x):", f_b1)
assert f_b1[2] == 0.0

▶ What you'll see: the loss is smallest at `x=3` and grows away from it.

👀 Takeaway: optimization starts by naming the scalar quantity we want to make small.

### Basic 2 — Compute a derivative by formula

**Goal.** Use $f'(x)=2(x-3)$, because the gradient is the slope used by descent.

In [ ]:
x_b2 = 0.0
grad_b2 = 2 * (x_b2 - 3)
print("x:", x_b2, "gradient:", grad_b2)
assert grad_b2 == -6.0

▶ What you'll see: the slope at zero is negative.

👀 Takeaway: the sign of the gradient tells which direction is locally uphill.

### Basic 3 — Move opposite the derivative

**Goal.** Take one update $x\leftarrow x-\eta f'(x)$, because descent negates the uphill direction.

In [ ]:
x_b3 = 0.0
eta_b3 = 0.1
grad_b3 = 2 * (x_b3 - 3)
x_next_b3 = x_b3 - eta_b3 * grad_b3
print("next x:", x_next_b3)
assert round(x_next_b3, 3) == 0.6

▶ What you'll see: subtracting a negative gradient moves `x` to the right.

👀 Takeaway: gradient descent is “current point minus step size times gradient.”

### Basic 4 — Verify the loss decreased

**Goal.** Compare loss before and after one step, because the update should improve the objective when the step is safe.

In [ ]:
x0_b4 = 0.0
x1_b4 = 0.6
loss0_b4 = (x0_b4 - 3) ** 2
loss1_b4 = (x1_b4 - 3) ** 2
print("loss before:", loss0_b4, "loss after:", loss1_b4)
assert loss1_b4 < loss0_b4 and round(loss1_b4, 2) == 5.76

▶ What you'll see: the loss falls from `9.0` to `5.76`.

👀 Takeaway: one good gradient step should be checked by actual objective decrease.

### Basic 5 — Run five gradient steps

**Goal.** Repeat the update a few times, because optimization is accumulated local improvement.

In [ ]:
eta_b5 = 0.2
xs_b5 = [0.0]
for step_b5 in range(5):
    grad_b5 = 2 * (xs_b5[-1] - 3)
    xs_b5.append(xs_b5[-1] - eta_b5 * grad_b5)
xs_b5 = np.array(xs_b5)
print("path:", np.round(xs_b5, 3))
assert np.all(np.diff((xs_b5 - 3) ** 2) < 0)

▶ What you'll see: each iterate gets closer to `3`.

👀 Takeaway: safe repeated steps contract the distance to the minimizer.

### Basic 6 — Plot the path on the bowl

**Goal.** Visualize iterates on the objective, because plots reveal whether descent is moving sensibly.

In [ ]:
grid_b6 = np.linspace(-1, 5, 200)
loss_grid_b6 = (grid_b6 - 3) ** 2
xs_b6 = np.array([0.0, 1.2, 1.92, 2.352, 2.6112])
plt.figure(figsize=(4.6, 3))
plt.plot(grid_b6, loss_grid_b6, color="navy")
plt.scatter(xs_b6, (xs_b6 - 3) ** 2, color="crimson")
plt.title("Basic 6: iterates on the quadratic")
plt.xlabel("x"); plt.ylabel("f(x)"); plt.show()

▶ What you'll see: the red points walk down the bowl toward the minimum.

👀 Takeaway: a trajectory plot is a simple debugging tool for descent behavior.

### Basic 7 — Compute the geometric error factor

**Goal.** Derive the contraction factor for the quadratic, because it predicts convergence speed.

In [ ]:
eta_b7 = 0.2
factor_b7 = 1 - 2 * eta_b7
error0_b7 = -3.0
error5_b7 = (factor_b7 ** 5) * error0_b7
print("factor:", factor_b7, "error after 5 steps:", round(error5_b7, 4))
assert round(factor_b7, 3) == 0.6

▶ What you'll see: with `η=0.2`, the error keeps 60% of its previous value each step.

👀 Takeaway: on a quadratic, step size directly controls the contraction factor.

### Basic 8 — Show overshooting with a large step

**Goal.** Try a too-large learning rate, because correct directions can still diverge.

In [ ]:
eta_b8 = 1.1
xs_b8 = [0.0]
for step_b8 in range(4):
    grad_b8 = 2 * (xs_b8[-1] - 3)
    xs_b8.append(xs_b8[-1] - eta_b8 * grad_b8)
xs_b8 = np.array(xs_b8)
print("large-step path:", np.round(xs_b8, 3))
assert abs(xs_b8[-1] - 3) > abs(xs_b8[0] - 3)

▶ What you'll see: the sequence crosses back and forth with growing amplitude.

👀 Takeaway: learning rate is a stability choice, not just a speed choice.

### Basic 9 — Update a two-parameter vector

**Goal.** Apply gradient descent to a vector parameter, because ML models usually have many parameters.

In [ ]:
w_b9 = np.array([-2.0, 1.0])
grad_b9 = np.array([2 * (w_b9[0] - 1), 8 * (w_b9[1] + 2)])
eta_b9 = 0.1
w_next_b9 = w_b9 - eta_b9 * grad_b9
print("gradient:", grad_b9, "next w:", w_next_b9)
assert np.allclose(w_next_b9, [-1.4, -1.4])

▶ What you'll see: each coordinate is updated by its own partial derivative.

👀 Takeaway: the gradient has the same shape as the parameter vector.

### Basic 10 — One regression-gradient step

**Goal.** Compute the first linear-regression update, because the same descent rule trains models.

In [ ]:
X_b10 = np.array([1.0, 2.0, 3.0])
y_b10 = 2 * X_b10
w_b10 = 0.0
grad_b10 = np.mean(2 * X_b10 * (w_b10 * X_b10 - y_b10))
w_next_b10 = w_b10 - 0.05 * grad_b10
print("gradient:", round(grad_b10, 3), "next w:", round(w_next_b10, 3))
assert round(w_next_b10, 3) == 0.933

▶ What you'll see: the slope jumps upward because predictions are too small.

👀 Takeaway: regression training is gradient descent on average squared prediction error.

## 🟡 Easy

### Easy 1 — Plot a full convergence curve

**Goal.** Track loss over many iterations, because convergence should be visible as a falling curve.

In [ ]:
eta_e1 = 0.2
x_e1 = 0.0
losses_e1 = []
for step_e1 in range(25):
    losses_e1.append((x_e1 - 3) ** 2)
    grad_e1 = 2 * (x_e1 - 3)
    x_e1 = x_e1 - eta_e1 * grad_e1
print("final x:", round(x_e1, 6), "final loss:", round((x_e1 - 3) ** 2, 10))
assert losses_e1[-1] < losses_e1[0]

▶ What you'll see: the final point is extremely close to the minimizer.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(losses_e1, marker="o", color="teal")
plt.yscale("log")
plt.title("Easy 1: gradient descent convergence")
plt.xlabel("iteration"); plt.ylabel("loss, log scale"); plt.show()

▶ What you'll see: a smoothly decreasing loss curve.

👀 Takeaway: convergence diagnostics turn repeated updates into inspectable evidence.

### Easy 2 — Compare several learning rates

**Goal.** Sweep step sizes, because the same gradient formula can be slow, fast, or unstable.

In [ ]:
etas_e2 = np.array([0.05, 0.2, 0.8, 1.1])
final_losses_e2 = []
paths_e2 = []
for eta_e2 in etas_e2:
    x_e2 = 0.0
    path_e2 = []
    for step_e2 in range(15):
        path_e2.append((x_e2 - 3) ** 2)
        x_e2 = x_e2 - eta_e2 * 2 * (x_e2 - 3)
    paths_e2.append(path_e2)
    final_losses_e2.append((x_e2 - 3) ** 2)
print("final losses:", np.round(final_losses_e2, 3))
assert final_losses_e2[-1] > final_losses_e2[0]

▶ What you'll see: moderate learning rates beat tiny ones, while `1.1` is unstable.

In [ ]:
plt.figure(figsize=(5, 3))
for eta_e2, path_e2 in zip(etas_e2, paths_e2):
    plt.plot(path_e2, label=f"η={eta_e2}")
plt.yscale("log"); plt.title("Easy 2: learning-rate sweep")
plt.xlabel("iteration"); plt.ylabel("loss"); plt.legend(); plt.show()

▶ What you'll see: the stable curves fall; the too-large curve rises.

👀 Takeaway: learning rate is the main knob controlling progress versus overshoot.

### Easy 3 — Train one-parameter linear regression

**Goal.** Fit $\hat y=wx$ by gradient descent, because model training uses the same update on data loss.

In [ ]:
X_e3 = np.array([1.0, 2.0, 3.0, 4.0])
y_e3 = 2.0 * X_e3
w_e3 = 0.0
losses_e3 = []
for step_e3 in range(60):
    pred_e3 = w_e3 * X_e3
    losses_e3.append(np.mean((pred_e3 - y_e3) ** 2))
    grad_e3 = np.mean(2 * X_e3 * (pred_e3 - y_e3))
    w_e3 = w_e3 - 0.03 * grad_e3
print("learned w:", round(w_e3, 4), "start/end loss:", round(losses_e3[0], 3), round(losses_e3[-1], 6))
assert abs(w_e3 - 2.0) < 0.01

▶ What you'll see: the learned slope approaches the true value `2.0`.

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.plot(losses_e3, color="purple")
plt.title("Easy 3: regression loss falls")
plt.xlabel("step"); plt.ylabel("MSE"); plt.show()

▶ What you'll see: MSE decreases toward zero as the line fits the data.

👀 Takeaway: gradients average per-example residual information into a parameter update.

### Easy 4 — See feature scaling change gradient size

**Goal.** Compare gradients before and after rescaling inputs, because feature scale changes safe learning rates.

In [ ]:
X_raw_e4 = np.array([10.0, 20.0, 30.0])
y_raw_e4 = 2 * X_raw_e4
w_e4 = 0.0
grad_raw_e4 = np.mean(2 * X_raw_e4 * (w_e4 * X_raw_e4 - y_raw_e4))
X_scaled_e4 = X_raw_e4 / 10.0
y_scaled_e4 = 2 * X_scaled_e4
grad_scaled_e4 = np.mean(2 * X_scaled_e4 * (w_e4 * X_scaled_e4 - y_scaled_e4))
print("raw gradient:", round(grad_raw_e4, 3), "scaled gradient:", round(grad_scaled_e4, 3))
assert abs(grad_raw_e4 / grad_scaled_e4 - 100) < 1e-9

▶ What you'll see: scaling `x` by 10 changes this gradient magnitude by 100.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["raw", "scaled"], [abs(grad_raw_e4), abs(grad_scaled_e4)], color=["crimson", "seagreen"])
plt.yscale("log")
plt.title("Easy 4: scale changes gradient magnitude")
plt.ylabel("|gradient|, log scale"); plt.show()

▶ What you'll see: the raw feature produces a much larger gradient.

👀 Takeaway: rescaling inputs can turn an unsafe learning rate into a safe one.

### Easy 5 — Use a gradient-norm stopping rule

**Goal.** Stop when the gradient is small, because near a smooth minimum the slope should approach zero.

In [ ]:
x_e5 = 0.0
eta_e5 = 0.2
tol_e5 = 1e-3
steps_e5 = 0
while abs(2 * (x_e5 - 3)) > tol_e5 and steps_e5 < 1000:
    x_e5 = x_e5 - eta_e5 * 2 * (x_e5 - 3)
    steps_e5 += 1
print("steps:", steps_e5, "x:", round(x_e5, 6), "gradient:", round(2 * (x_e5 - 3), 8))
assert abs(2 * (x_e5 - 3)) <= tol_e5

▶ What you'll see: the loop stops once the local slope is tiny.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["distance to optimum", "|gradient|"], [abs(x_e5 - 3), abs(2 * (x_e5 - 3))], color="teal")
plt.title("Easy 5: stopping quantities")
plt.show()

▶ What you'll see: both the distance and gradient norm are very small on this convex quadratic.

👀 Takeaway: small gradients are useful stopping signals, especially in convex smooth problems.

## 🔴 Advanced

### Advanced 1 — Descend an ill-conditioned bowl

**Goal.** Optimize an elliptical quadratic, because different curvatures make one learning rate behave unevenly across coordinates.

In [ ]:
w_a1 = np.array([5.0, 5.0])
eta_a1 = 0.08
path_a1 = [w_a1.copy()]
for step_a1 in range(40):
    grad_a1 = np.array([2 * w_a1[0], 20 * w_a1[1]])
    w_a1 = w_a1 - eta_a1 * grad_a1
    path_a1.append(w_a1.copy())
path_a1 = np.array(path_a1)
loss_a1 = path_a1[:, 0] ** 2 + 10 * path_a1[:, 1] ** 2
print("start/end loss:", round(loss_a1[0], 3), round(loss_a1[-1], 6))
assert loss_a1[-1] < loss_a1[0]

▶ What you'll see: the loss decreases, but the steep coordinate can oscillate.

In [ ]:
plt.figure(figsize=(4.8, 3.5))
plt.plot(path_a1[:, 0], path_a1[:, 1], marker="o", markersize=3, color="crimson")
plt.title("Advanced 1: path on ill-conditioned bowl")
plt.xlabel("w0"); plt.ylabel("w1"); plt.show()

▶ What you'll see: the path zigzags because one coordinate has much larger curvature.

👀 Takeaway: ill-conditioning makes one global step size a compromise across coordinates.

### Advanced 2 — Compare batch and stochastic gradients

**Goal.** Contrast full-data and one-example gradients, because stochastic optimization replaces exact gradients with noisy estimates.

In [ ]:
X_a2 = np.array([1.0, 2.0, 3.0, 4.0])
y_a2 = 2 * X_a2
w_batch_a2 = 0.0
w_sgd_a2 = 0.0
eta_a2 = 0.03
batch_losses_a2 = []
sgd_losses_a2 = []
for step_a2 in range(80):
    pred_batch_a2 = w_batch_a2 * X_a2
    grad_batch_a2 = np.mean(2 * X_a2 * (pred_batch_a2 - y_a2))
    w_batch_a2 -= eta_a2 * grad_batch_a2
    i_a2 = step_a2 % len(X_a2)
    grad_sgd_a2 = 2 * X_a2[i_a2] * (w_sgd_a2 * X_a2[i_a2] - y_a2[i_a2])
    w_sgd_a2 -= eta_a2 * grad_sgd_a2
    batch_losses_a2.append(np.mean((w_batch_a2 * X_a2 - y_a2) ** 2))
    sgd_losses_a2.append(np.mean((w_sgd_a2 * X_a2 - y_a2) ** 2))
print("batch w:", round(w_batch_a2, 3), "sgd w:", round(w_sgd_a2, 3))
assert abs(w_batch_a2 - 2) < 0.01 and abs(w_sgd_a2 - 2) < 0.05

▶ What you'll see: both approaches learn the slope, but SGD takes noisier steps.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(batch_losses_a2, label="batch")
plt.plot(sgd_losses_a2, label="stochastic", alpha=0.8)
plt.yscale("log")
plt.title("Advanced 2: exact vs noisy gradients")
plt.xlabel("update"); plt.ylabel("MSE"); plt.legend(); plt.show()

▶ What you'll see: the stochastic curve is less smooth but still trends downward.

👀 Takeaway: noisy gradients can target the same optimum while reducing per-step cost.

### Advanced 3 — Use momentum to smooth zigzags

**Goal.** Add a velocity term, because momentum accumulates consistent gradient directions and damps alternating ones.

In [ ]:
w_plain_a3 = np.array([5.0, 5.0])
w_mom_a3 = np.array([5.0, 5.0])
v_a3 = np.zeros(2)
eta_a3 = 0.08
beta_a3 = 0.8
loss_plain_a3 = []
loss_mom_a3 = []
for step_a3 in range(60):
    grad_plain_a3 = np.array([2 * w_plain_a3[0], 20 * w_plain_a3[1]])
    w_plain_a3 -= eta_a3 * grad_plain_a3
    grad_mom_a3 = np.array([2 * w_mom_a3[0], 20 * w_mom_a3[1]])
    v_a3 = beta_a3 * v_a3 + grad_mom_a3
    w_mom_a3 -= eta_a3 * v_a3
    loss_plain_a3.append(w_plain_a3[0] ** 2 + 10 * w_plain_a3[1] ** 2)
    loss_mom_a3.append(w_mom_a3[0] ** 2 + 10 * w_mom_a3[1] ** 2)
print("final plain/momentum loss:", round(loss_plain_a3[-1], 5), round(loss_mom_a3[-1], 5))
assert np.isfinite(loss_mom_a3[-1])

▶ What you'll see: momentum changes the trajectory by using accumulated past gradients.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(loss_plain_a3, label="plain GD")
plt.plot(loss_mom_a3, label="momentum")
plt.yscale("log")
plt.title("Advanced 3: momentum changes convergence")
plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

▶ What you'll see: momentum can accelerate broad directions but may oscillate if too aggressive.

👀 Takeaway: optimizer variants still rely on gradients, but modify how steps are accumulated.

### Advanced 4 — Detect a saddle with a small gradient

**Goal.** Inspect $f(x,y)=x^2-y^2$, because a zero gradient is not always a minimum.

In [ ]:
point_a4 = np.array([0.0, 0.0])
grad_a4 = np.array([2 * point_a4[0], -2 * point_a4[1]])
H_a4 = np.array([[2.0, 0.0], [0.0, -2.0]])
eigs_a4 = np.linalg.eigvalsh(H_a4)
print("gradient:", grad_a4, "Hessian eigenvalues:", eigs_a4)
assert np.allclose(grad_a4, [0, 0]) and eigs_a4[0] < 0 < eigs_a4[1]

▶ What you'll see: the gradient is zero, but curvature is positive in one direction and negative in another.

In [ ]:
x_a4 = np.linspace(-2, 2, 80)
y_a4 = np.linspace(-2, 2, 80)
X_a4, Y_a4 = np.meshgrid(x_a4, y_a4)
Z_a4 = X_a4 ** 2 - Y_a4 ** 2
plt.figure(figsize=(4.8, 3.5))
plt.contour(X_a4, Y_a4, Z_a4, levels=21, cmap="coolwarm")
plt.scatter([0], [0], color="black")
plt.title("Advanced 4: zero-gradient saddle")
plt.xlabel("x"); plt.ylabel("y"); plt.show()

▶ What you'll see: contours bend upward along one axis and downward along the other.

👀 Takeaway: small gradients alone do not prove optimality in nonconvex problems.

### Advanced 5 — Gradient descent with standardized features

**Goal.** Fit two-feature regression before and after standardization, because scaling improves conditioning and learning-rate safety.

In [ ]:
X_raw_a5 = np.array([[1.0, 100.0], [2.0, 200.0], [3.0, 300.0], [4.0, 400.0]])
y_a5 = np.array([3.0, 6.0, 9.0, 12.0])
mean_a5 = X_raw_a5.mean(axis=0)
std_a5 = X_raw_a5.std(axis=0)
X_a5 = (X_raw_a5 - mean_a5) / std_a5
w_a5 = np.zeros(2)
b_a5 = 0.0
losses_a5 = []
for step_a5 in range(100):
    pred_a5 = X_a5 @ w_a5 + b_a5
    err_a5 = pred_a5 - y_a5
    losses_a5.append(np.mean(err_a5 ** 2))
    grad_w_a5 = (2 / len(y_a5)) * (X_a5.T @ err_a5)
    grad_b_a5 = 2 * np.mean(err_a5)
    w_a5 -= 0.1 * grad_w_a5
    b_a5 -= 0.1 * grad_b_a5
print("final loss:", round(losses_a5[-1], 8), "bias:", round(b_a5, 3), "weights:", np.round(w_a5, 3))
assert losses_a5[-1] < 1e-6

▶ What you'll see: standardized features allow a simple learning rate to drive loss nearly to zero.

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(losses_a5, color="seagreen")
plt.yscale("log")
plt.title("Advanced 5: standardized-feature descent")
plt.xlabel("step"); plt.ylabel("MSE"); plt.show()

▶ What you'll see: the loss falls smoothly despite the raw features living on very different scales.

👀 Takeaway: feature standardization improves the geometry that gradient descent sees.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Gradient descent repeats one humble act: subtract a step size times the gradient, and let local slope accumulate into global progress.

Gradient descent operationalizes first-order optimality. The direction is simple, but curvature and feature scale decide whether a fixed step converges or explodes. Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

SEED = 20260701
rng = np.random.default_rng(SEED)


def quadratic_surface(name, H, center, x0, project=None):
    H = np.asarray(H, dtype=float)
    center = np.asarray(center, dtype=float)
    x0 = np.asarray(x0, dtype=float)

    def loss(x):
        z = np.asarray(x, dtype=float) - center
        return 0.5 * float(z @ H @ z)

    def grad(x):
        z = np.asarray(x, dtype=float) - center
        return H @ z

    def hess(x):
        return H

    return {
        "name": name,
        "kind": "quadratic",
        "dim": len(x0),
        "x0": x0,
        "loss": loss,
        "grad": grad,
        "hess": hess,
        "project": project,
        "center": center,
    }


def rosenbrock_surface():
    def loss(x):
        x = np.asarray(x, dtype=float)
        a = x[0]
        b = x[1]
        return float(100.0 * (b - a * a) ** 2 + (1.0 - a) ** 2)

    def grad(x):
        x = np.asarray(x, dtype=float)
        a = x[0]
        b = x[1]
        return np.array([
            -400.0 * a * (b - a * a) - 2.0 * (1.0 - a),
            200.0 * (b - a * a),
        ])

    def hess(x):
        x = np.asarray(x, dtype=float)
        a = x[0]
        b = x[1]
        return np.array([
            [1200.0 * a * a - 400.0 * b + 2.0, -400.0 * a],
            [-400.0 * a, 200.0],
        ])

    return {
        "name": "D3 nonconvex Rosenbrock valley",
        "kind": "rosenbrock",
        "dim": 2,
        "x0": np.array([-1.2, 1.0]),
        "loss": loss,
        "grad": grad,
        "hess": hess,
        "project": None,
        "center": np.array([1.0, 1.0]),
    }


def logistic_surface():
    data = load_breast_cancer()
    X = StandardScaler().fit_transform(data.data)
    X = np.column_stack([np.ones(X.shape[0]), X])
    y = data.target.astype(float)
    reg = 0.05

    def loss(w):
        z = X @ w
        yz = y * z
        logistic = np.logaddexp(0.0, z) - yz
        penalty = 0.5 * reg * float(w[1:] @ w[1:])
        return float(np.mean(logistic) + penalty)

    def grad(w):
        z = X @ w
        p = 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
        g = X.T @ (p - y) / X.shape[0]
        g[1:] = g[1:] + reg * w[1:]
        return g

    def hess(w):
        z = X @ w
        p = 1.0 / (1.0 + np.exp(-np.clip(z, -40.0, 40.0)))
        weights = p * (1.0 - p)
        Xw = X * weights[:, None]
        H = X.T @ Xw / X.shape[0]
        H[1:, 1:] = H[1:, 1:] + reg * np.eye(X.shape[1] - 1)
        return H

    return {
        "name": "D4 breast-cancer logistic loss",
        "kind": "logistic",
        "dim": X.shape[1],
        "x0": np.zeros(X.shape[1]),
        "loss": loss,
        "grad": grad,
        "hess": hess,
        "project": None,
        "center": np.zeros(X.shape[1]),
        "X_shape": X.shape,
        "classes": sorted(set(data.target.tolist())),
    }


def make_high_dim_box_surface(dim=20):
    Q, _ = np.linalg.qr(rng.normal(size=(dim, dim)))
    eigs = np.geomspace(1.0, 120.0, dim)
    H = Q @ np.diag(eigs) @ Q.T
    center = np.linspace(-0.8, 0.8, dim)
    x0 = np.linspace(1.4, -1.4, dim)

    def project(x):
        return np.clip(x, -1.0, 1.0)

    return quadratic_surface("D5 high-dimensional box-constrained SPD bowl", H, center, x0, project=project)


def make_loss_ladder():
    d1 = quadratic_surface(
        "D1 2-D quadratic bowl",
        np.diag([2.0, 4.0]),
        np.array([1.0, -1.0]),
        np.array([-2.0, 2.0]),
    )
    d2 = quadratic_surface(
        "D2 anisotropic ill-conditioned quadratic",
        np.diag([1.0, 80.0]),
        np.array([-1.0, 1.0]),
        np.array([2.5, -2.0]),
    )
    return [d1, d2, rosenbrock_surface(), logistic_surface(), make_high_dim_box_surface()]


def project_if_needed(surface, x):
    if surface.get("project") is None:
        return np.asarray(x, dtype=float)
    return surface["project"](np.asarray(x, dtype=float))


def run_optimizer(surface, steps=120, eta=0.05, tol=1e-6):
    x = project_if_needed(surface, surface["x0"])
    losses = []
    path = [x.copy()]
    for step in range(steps):
        loss_value = surface["loss"](x)
        losses.append(loss_value)
        g = surface["grad"](x)
        if np.linalg.norm(g) < tol:
            break
        trial = x - eta * g
        x = project_if_needed(surface, trial)
        path.append(x.copy())
    losses.append(surface["loss"](x))
    return {
        "x": x,
        "losses": np.array(losses),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def backtracking_line_search(loss, grad, x, direction, alpha0=1.0, rho=0.5, c=0.5):
    alpha = alpha0
    f0 = loss(x)
    g0 = grad(x)
    slope = float(g0 @ direction)
    while loss(x + alpha * direction) > f0 + c * alpha * slope:
        alpha = rho * alpha
        if alpha < 1e-10:
            break
    return alpha


def line_search_optimizer(surface, steps=80, tol=1e-6):
    x = project_if_needed(surface, surface["x0"])
    losses = []
    path = [x.copy()]
    for step in range(steps):
        losses.append(surface["loss"](x))
        g = surface["grad"](x)
        if np.linalg.norm(g) < tol:
            break
        direction = -g
        alpha = backtracking_line_search(surface["loss"], surface["grad"], x, direction)
        x = project_if_needed(surface, x + alpha * direction)
        path.append(x.copy())
    losses.append(surface["loss"](x))
    return {
        "x": x,
        "losses": np.array(losses),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def bfgs_update(B, s, y):
    Bs = B @ s
    sBs = float(s @ Bs)
    ys = float(y @ s)
    if ys <= 1e-12 or sBs <= 1e-12:
        return B
    return B - np.outer(Bs, Bs) / sBs + np.outer(y, y) / ys


def damped_newton_optimizer(surface, steps=50, tol=1e-6, damping=1e-4):
    x = project_if_needed(surface, surface["x0"])
    losses = []
    path = [x.copy()]
    for step in range(steps):
        losses.append(surface["loss"](x))
        g = surface["grad"](x)
        if np.linalg.norm(g) < tol:
            break
        H = surface["hess"](x)
        shift = damping
        eye = np.eye(len(x))
        while np.min(np.linalg.eigvalsh(H + shift * eye)) <= 0.0:
            shift = 10.0 * shift
        direction = -np.linalg.solve(H + shift * eye, g)
        if float(direction @ g) >= 0.0:
            direction = -g
        alpha = backtracking_line_search(surface["loss"], surface["grad"], x, direction, c=1e-4)
        x = project_if_needed(surface, x + alpha * direction)
        path.append(x.copy())
    losses.append(surface["loss"](x))
    return {
        "x": x,
        "losses": np.array(losses),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def bfgs_optimizer(surface, steps=60, tol=1e-6):
    x = project_if_needed(surface, surface["x0"])
    dim = len(x)
    B = np.eye(dim)
    losses = []
    path = [x.copy()]
    for step in range(steps):
        losses.append(surface["loss"](x))
        g = surface["grad"](x)
        if np.linalg.norm(g) < tol:
            break
        direction = -np.linalg.solve(B + 1e-8 * np.eye(dim), g)
        if float(direction @ g) >= 0.0:
            direction = -g
        alpha = backtracking_line_search(surface["loss"], surface["grad"], x, direction, c=1e-4)
        x_next = project_if_needed(surface, x + alpha * direction)
        s = x_next - x
        y = surface["grad"](x_next) - g
        B = bfgs_update(B, s, y)
        x = x_next
        path.append(x.copy())
    losses.append(surface["loss"](x))
    return {
        "x": x,
        "losses": np.array(losses),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def conjugate_gradient(A, b, x0=None, tol=1e-8, max_iter=None):
    A = np.asarray(A, dtype=float)
    b = np.asarray(b, dtype=float)
    if x0 is None:
        x = np.zeros_like(b)
    else:
        x = np.asarray(x0, dtype=float).copy()
    if max_iter is None:
        max_iter = len(b)
    r = b - A @ x
    p = r.copy()
    rs_old = float(r @ r)
    residuals = [np.sqrt(rs_old)]
    path = [x.copy()]
    for k in range(max_iter):
        Ap = A @ p
        denom = float(p @ Ap)
        if denom <= 0.0:
            raise ValueError("CG requires symmetric positive-definite curvature")
        alpha = rs_old / denom
        x = x + alpha * p
        r = r - alpha * Ap
        rs_new = float(r @ r)
        residuals.append(np.sqrt(rs_new))
        path.append(x.copy())
        if np.sqrt(rs_new) < tol:
            break
        beta = rs_new / rs_old
        p = r + beta * p
        rs_old = rs_new
    return {
        "x": x,
        "residuals": np.array(residuals),
        "path": np.array(path),
        "iterations": len(path) - 1,
    }


def make_cg_systems():
    systems = []
    base = make_loss_ladder()
    for surface in base:
        if surface["kind"] == "rosenbrock":
            point = np.array([1.0, 1.0])
            A = surface["hess"](point) + 1e-3 * np.eye(2)
            b = A @ point
            name = "D3 Rosenbrock SPD local Newton system"
        elif surface["kind"] == "logistic":
            point = surface["x0"]
            A = surface["hess"](point)
            b = -surface["grad"](point)
            name = "D4 logistic Hessian-vector SPD system"
        else:
            A = surface["hess"](surface["x0"])
            b = A @ surface["center"]
            name = surface["name"].replace("loss", "system")
        systems.append({
            "name": name,
            "A": A,
            "b": b,
            "x0": np.zeros_like(b),
        })
    return systems


def preview_ladder(ladder):
    for idx, item in enumerate(ladder, start=1):
        if "A" in item:
            shape = item["A"].shape
            extra = f"system_shape={shape}"
            sample = item["b"][: min(3, len(item["b"]))]
        else:
            shape = (item["dim"],)
            extra = f"dim={item['dim']} kind={item['kind']}"
            sample = item["x0"][: min(3, len(item["x0"]))]
        print(f"D{idx}: {item['name']} | {extra} | sample={sample}")


def plot_loss_contours(ax, surface, path, title):
    path = np.asarray(path)
    if path.ndim == 1:
        path = path[:, None]
    if path.shape[1] == 1:
        xs = np.linspace(-3.0, 3.0, 120)
        ys = np.zeros_like(xs)
        vals = np.array([surface["loss"](np.array([x])) for x in xs])
        ax.plot(xs, vals)
        ax.set_title(title)
        return
    x_min = min(np.min(path[:, 0]) - 0.5, -2.5)
    x_max = max(np.max(path[:, 0]) + 0.5, 2.5)
    y_min = min(np.min(path[:, 1]) - 0.5, -2.5)
    y_max = max(np.max(path[:, 1]) + 0.5, 2.5)
    xs = np.linspace(x_min, x_max, 100)
    ys = np.linspace(y_min, y_max, 100)
    xx, yy = np.meshgrid(xs, ys)
    base = path[-1].copy()
    zz = np.zeros_like(xx)
    for i in range(xx.shape[0]):
        for j in range(xx.shape[1]):
            point = base.copy()
            point[0] = xx[i, j]
            point[1] = yy[i, j]
            zz[i, j] = surface["loss"](point)
    ax.contour(xx, yy, zz, levels=20)
    ax.plot(path[:, 0], path[:, 1], marker="o", markersize=2)
    ax.set_title(title)


def plot_system_contours(ax, system, path, title):
    A = system["A"]
    b = system["b"]
    path = np.asarray(path)
    if path.shape[1] < 2:
        ax.plot(np.arange(len(path)), path[:, 0])
        ax.set_title(title)
        return
    x_min = min(np.min(path[:, 0]) - 0.5, -2.0)
    x_max = max(np.max(path[:, 0]) + 0.5, 2.0)
    y_min = min(np.min(path[:, 1]) - 0.5, -2.0)
    y_max = max(np.max(path[:, 1]) + 0.5, 2.0)
    xs = np.linspace(x_min, x_max, 90)
    ys = np.linspace(y_min, y_max, 90)
    xx, yy = np.meshgrid(xs, ys)
    base = path[-1].copy()
    zz = np.zeros_like(xx)
    for i in range(xx.shape[0]):
        for j in range(xx.shape[1]):
            point = base.copy()
            point[0] = xx[i, j]
            point[1] = yy[i, j]
            zz[i, j] = 0.5 * float(point @ A @ point) - float(b @ point)
    ax.contour(xx, yy, zz, levels=20)
    ax.plot(path[:, 0], path[:, 1], marker="o", markersize=2)
    ax.set_title(title)

## The concept, built once (D1)

The update is $x_{t+1}=x_t-\eta\nabla f(x_t)$, with $\eta\gt0$ chosen small enough for the local curvature.

The D1 check is intentionally tiny: it plugs in the lesson's exact numbers before the same optimizer machinery is used on harder rungs.

In [ ]:
def gradient_descent(grad, x0, eta, steps):
    x = float(x0)
    path = [x]
    for step in range(steps):
        x = x - eta * grad(x)
        path.append(x)
    return np.array(path)

grad = lambda x: 2.0 * (x - 3.0)
path = gradient_descent(grad, 0.0, 0.1, 1)
assert np.isclose(grad(0.0), -6.0)
assert np.isclose(path[1], 0.6)
print("one-step path:", path)

The same update should lower the toy loss from $9$ to $5.76$ and obey the lesson's contraction rule when $0\lt\eta\lt1$ for curvature $L=2$.

In [ ]:
loss0 = (0.0 - 3.0) ** 2
loss1 = (0.6 - 3.0) ** 2
safe_multiplier = 1.0 - 2.0 * 0.2
error_after_20 = 3.0 * safe_multiplier ** 20
assert np.isclose(loss0, 9.0)
assert np.isclose(loss1, 5.76)
assert np.isclose(safe_multiplier, 0.6)
assert np.isclose(error_after_20, 0.0001097, atol=1e-7)
print("loss drop:", loss0, "to", loss1)
print("20-step error:", error_after_20)

## Dataset ladder: D1→D5 loss surfaces

Inline F4 ladder, no shared helper import: D1 quadratic bowl, D2 ill-conditioned quadratic, D3 Rosenbrock/nonconvex, D4 real `load_breast_cancer` logistic loss, and D5 high-dimensional or constrained. The metric is final loss.

In [ ]:
ladder = make_loss_ladder()
preview_ladder(ladder)

## Run the same method across D1→D5

Only the method for this lesson changes; the ladder stays fixed. Collect one metric per rung: final loss.

In [ ]:
ladder = make_loss_ladder()
results = []
for rung, surface in enumerate(ladder, start=1):
    eta = 0.02 if rung in [2, 3, 5] else 0.05
    if surface["kind"] == "logistic":
        eta = 0.2
    result = run_optimizer(surface, eta=eta, steps=180)
    results.append(result)
    print(f"D{rung} | eta={eta:.3f} | final_loss={result['losses'][-1]:.6f}")

## Results visualization

The closing figure has two parts: optimizer trajectories on contour panels, then the metric curve across D1→D5.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 3.5))
for idx, (surface, result) in enumerate(zip(ladder, results)):
    plot_loss_contours(axes[idx], surface, result["path"], f"D{idx + 1}")
plt.tight_layout()
plt.show()

metrics = [result["losses"][-1] for result in results]
plt.figure(figsize=(6, 3.5))
plt.plot(range(1, 6), metrics, marker="o")
plt.xlabel("rung")
plt.ylabel("final loss")
plt.title("Final loss across the ladder")
plt.grid(True)
plt.show()

## Pitfall on the hardest rung

Pitfall on D5: one step size for all curvature. The same learning rate that is safe on a round bowl can bounce on ill-conditioned or high-dimensional features. Scaling or an adaptive smaller step fixes the behavior.

In [ ]:
surface = make_loss_ladder()[4]
large_step = run_optimizer(surface, eta=0.08, steps=30)
small_step = run_optimizer(surface, eta=0.008, steps=30)
print("large-step final loss:", large_step["losses"][-1])
print("small-step final loss:", small_step["losses"][-1])
print("improved by smaller/scaled step:", small_step["losses"][-1] <= large_step["losses"][-1])

## Evaluate it + Practice

- Metric: final loss; compare against a no-skill baseline that keeps the initial point or takes a fixed tiny step.
- Sanity check: on D1, verify the output against the closed-form minimizer or exact linear-system solution.
- Ablation: turn off the key safeguard or curvature idea and confirm the metric worsens.
- Failure signals: nondecreasing loss, nonpositive CG denominator, exploding path length, or a tiny gradient with bad curvature.
- Reproducibility: keep the seed fixed and do not download data; `load_breast_cancer` is bundled with sklearn.

Practice prompts:

1. Change one tolerance and predict which rung changes the most.

2. Replace D2's condition number and compare the metric curve.

3. Add one diagnostic print that would catch the pitfall earlier.